# GPU VRAM Benchmark — FP32 vs FP16 vs BF16

Runs on a GPU-enabled Kaggle/Colab session. Measures how much VRAM a small model actually uses at each precision, to build intuition before scaling up to real fine-tuning runs.

**Before running:** on Kaggle, enable **Settings → Accelerator → GPU T4 x2**. Do not pick P100 — modern PyTorch wheels (2.x + cu12x) have dropped kernel support for the P100's older `sm_60` architecture, so CUDA calls on a P100 will hang or fail; T4 (`sm_75`) is fully supported.

## 1. Pull the repo and install dependencies

This repo is private, so an anonymous `git clone` will hang on a username/password prompt. Instead:

1. Create a GitHub Personal Access Token: github.com → Settings → Developer settings → Personal access tokens → Fine-grained tokens → generate one scoped to just this repo, with **Contents: Read-only** permission.
2. In this Kaggle notebook: **Add-ons → Secrets → Add a new secret**, name it `GITHUB_TOKEN`, paste the token value.
3. Run the cell below — it reads the secret and never prints it.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
repo_url = f"https://{github_token}@github.com/zoom-BT/llm-alignment-internship.git"

!git clone {repo_url}
%cd llm-alignment-internship
!pip install -q -r requirements.txt

## 2. Confirm the GPU is visible

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 3. Load the same model in FP32, FP16, and BF16, and record VRAM

`MODEL_NAME` is a small model picked only to make the *precision comparison* cheap to run — it is not the final base model decision for the internship (that's still open in `config.yaml`).

In [ ]:
import gc

from transformers import AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B"


def measure_vram(dtype, label):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to("cuda")
    allocated_mb = torch.cuda.memory_allocated() / 1024**2
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return {"precision": label, "allocated_mb": round(allocated_mb, 1), "peak_mb": round(peak_mb, 1)}


results = [
    measure_vram(torch.float32, "FP32"),
    measure_vram(torch.float16, "FP16"),
    measure_vram(torch.bfloat16, "BF16"),
]
results

## 4. Save the results as JSON

Written into `results/`, matching the repo's convention of tracking metrics as JSON (checkpoints/weights stay git-ignored, this file does not).

In [ ]:
import json
from pathlib import Path

Path("results").mkdir(exist_ok=True)
with open("results/vram_benchmark.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved to results/vram_benchmark.json")

## 5. Bring the results back into the local repo

Kaggle notebooks don't push to GitHub. Download `results/vram_benchmark.json` from the Kaggle output panel, drop it into the local `results/` folder, and commit it there.